# Activity 1: RAGAS Evaluation with Cost Analysis

Compare a **Fireworks AI** open-source RAG pipeline against an **OpenAI `gpt-4.1-mini`** equivalent on the same cat-health corpus and synthetic test set from Session 5.

**Metrics:** `ContextRecall` (retrieval), `Faithfulness`, `AnswerAccuracy`

**Instrumentation:** LangSmith tracing on both pipelines for per-query token usage and cost.

## Task 1: Environment Setup

From `10_LLM_Servers`:

```bash
uv sync
```

Ensure `.env` has `FIREWORKS_API_KEY`, `OPENAI_API_KEY`, `LANGSMITH_API_KEY`, and `LANGSMITH_TRACING=true`.

In [2]:
from __future__ import annotations

import asyncio
import os
from getpass import getpass
from pathlib import Path

import instructor
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import AnswerAccuracy, ContextRecall, Faithfulness
from ragas.run_config import RunConfig

from app.eval_helpers import (
    load_testset,
    run_rag_over_testset,
    run_ragas_async,
    run_ragas_sync,
)
from app.eval_rag import (
    build_provider_rag_pipeline,
    fireworks_config,
    openai_config,
)

load_dotenv()
load_dotenv(Path("..") / ".env")  # repo-root .env fallback


def read_required_secret(names: tuple[str, ...], prompt: str) -> str:
    for name in names:
        if value := os.environ.get(name):
            return value
    value = getpass(prompt)
    os.environ[names[0]] = value
    return value


read_required_secret(("FIREWORKS_API_KEY",), "Fireworks API key: ")
read_required_secret(("OPENAI_API_KEY",), "OpenAI API key: ")
read_required_secret(("LANGSMITH_API_KEY",), "LangSmith API key: ")

os.environ.setdefault("LANGSMITH_TRACING", "true")

DATA_DIR = os.environ.get("RAG_DATA_DIR", "data")
TESTSET_PATH = Path("artifacts/cat_health_synthetic_testset.jsonl")
EVAL_CASE_LIMIT = int(os.environ.get("EVAL_CASE_LIMIT", "4"))
JUDGE_MODEL_NAME = os.environ.get("OPENAI_JUDGE_MODEL", "gpt-4.1-mini")
OPENAI_BASE_URL = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1")

print(f"Corpus: {DATA_DIR}")
print(f"Test set: {TESTSET_PATH}")
print(f"Eval cases: {EVAL_CASE_LIMIT}")
print(f"Judge model: {JUDGE_MODEL_NAME}")

Corpus: data
Test set: artifacts/cat_health_synthetic_testset.jsonl
Eval cases: 4
Judge model: gpt-4.1-mini


## Task 2: Load Reviewed Synthetic Test Set

Reuse the curated examples from Session 5 — no regeneration needed.

In [3]:
testset_df = load_testset(TESTSET_PATH, limit=EVAL_CASE_LIMIT)
testset_df[["user_input", "reference"]]

,user_input,reference
0,What role does Paula Plummer have in the 2021 ...,Paula Plummer is listed among the authors of t...
1,What are the Feline Life Stages?,The feline patient’s life stage is the most fu...
2,"According to the feline life stage guidelines,...",The guidelines recommend starting consultation...
3,"How do the ""feline life stage healthcare"" idea...",The guidelines say the Task Force uses feline ...


## Task 3: Build Both RAG Pipelines

Same corpus, chunking, retriever (`k=3`), and prompt — only the provider changes.

In [4]:
fw_config = fireworks_config()
oa_config = openai_config()

print("Fireworks:", fw_config.chat_model, "|", fw_config.embedding_model)
print("OpenAI:", oa_config.chat_model, "|", oa_config.embedding_model)

# Tip: if your dedicated Fireworks deployment is scaled to zero, temporarily set
# FIREWORKS_CHAT_MODEL=accounts/fireworks/models/gpt-oss-20b in .env

Fireworks: accounts/majeedmo-8n5w6yco206/deployments/qb6oudta | accounts/fireworks/models/qwen3-embedding-8b
OpenAI: gpt-4.1-mini | text-embedding-3-small


## Task 4: Run Eval Queries with LangSmith Tracing

Set a **separate LangSmith project** per provider so cost dashboards are easy to compare.

In [5]:
def run_provider_eval(provider_name: str, config) -> list[dict]:
    os.environ["LANGSMITH_PROJECT"] = f"session10-rag-{provider_name}"
    graph = build_provider_rag_pipeline(DATA_DIR, config)
    rows = run_rag_over_testset(graph, testset_df)
    for row in rows:
        row["provider"] = provider_name
    return rows


fireworks_rows = run_provider_eval("fireworks", fw_config)
openai_rows = run_provider_eval("openai", oa_config)

pd.DataFrame(fireworks_rows)[["provider", "user_input", "response"]]

,provider,user_input,response
0,fireworks,What role does Paula Plummer have in the 2021 ...,Paula Plummer is listed as one of the authors ...
1,fireworks,What are the Feline Life Stages?,**Feline Life Stages (per the 2021 AAHA/AAFP F...
2,fireworks,"According to the feline life stage guidelines,...",**How the guidelines recommend combining open‑...
3,fireworks,"How do the ""feline life stage healthcare"" idea...",**How the guidelines help build a lifelong hea...


## Task 5: Score with RAGAS (3 metrics)

Use a fixed OpenAI judge so both providers are scored consistently.

In [8]:
def build_sync_judge_llm():
    judge = llm_factory(
        JUDGE_MODEL_NAME,
        provider="openai",
        client=OpenAI(api_key=os.environ["OPENAI_API_KEY"], base_url=OPENAI_BASE_URL),
        mode=instructor.Mode.TOOLS,
        max_tokens=1024,
    )
    judge.model_args = {"max_tokens": 2048, "max_retries": 3}

    async def agenerate_from_sync(prompt, response_model):
        return await asyncio.to_thread(
            judge.generate,
            prompt=prompt,
            response_model=response_model,
        )

    judge.agenerate = agenerate_from_sync
    return judge


async def score_rag_rows(rows: list[dict]) -> pd.DataFrame:
    judge_llm = build_sync_judge_llm()
    metrics = {
        "context_recall": ContextRecall(llm=judge_llm),
        "faithfulness": Faithfulness(llm=judge_llm),
        "answer_accuracy": AnswerAccuracy(llm=judge_llm),
    }

    score_rows = []
    for index, row in enumerate(rows, start=1):
        score_rows.append(
            {
                "case": index,
                "provider": row["provider"],
                "context_recall": (
                    await metrics["context_recall"].ascore(
                        user_input=row["user_input"],
                        retrieved_contexts=row["retrieved_contexts"],
                        reference=row["reference"],
                    )
                ).value,
                "faithfulness": (
                    await metrics["faithfulness"].ascore(
                        user_input=row["user_input"],
                        response=row["response"],
                        retrieved_contexts=row["retrieved_contexts"],
                    )
                ).value,
                "answer_accuracy": (
                    await metrics["answer_accuracy"].ascore(
                        user_input=row["user_input"],
                        response=row["response"],
                        reference=row["reference"],
                    )
                ).value,
            }
        )
    return pd.DataFrame(score_rows)


ragas_run_config = RunConfig(timeout=180, max_retries=3, max_wait=30, max_workers=2)

In [9]:
fireworks_scores = run_ragas_async(score_rag_rows, fireworks_rows)
openai_scores = run_ragas_async(score_rag_rows, openai_rows)

comparison = pd.concat([fireworks_scores, openai_scores], ignore_index=True)
comparison.groupby("provider").mean(numeric_only=True).T

provider,fireworks,openai
case,2.500000,2.500000
context_recall,0.687500,0.770833
faithfulness,0.777462,0.989583
answer_accuracy,0.437500,0.687500


## Task 6: LangSmith Cost Analysis & Quality Analysis

#### Cost Metrics (Extrapolate to 1,000 queries/day)
daily cost = per-query cost × 1,000
monthly cost ≈ daily cost × 30

| Provider  |	Per query |	1,000 queries/day |	~30 days  |
|-----------|-------------|-------------------|-----------|
| Fireworks | $0.0025     | $2.50/day         |~$75/mo    |
| OpenAI    | $0.0275     | $27.50/day        |~$825/mo   |

# Quality metrics

|Metric           |Fireworks |  OpenAI|   Winner|
|-----------------|----------|--------|---------|
|Context recall   | 0.69     |   0.77 |   OpenAI|
|Faithfulness     | 0.78     |   0.99 |   OpenAI|
|Answer accuracy  | 0.44     |   0.69 |   OpenAI|

OpenAI wins on all three quality metrics, but Fireworks is cheaper and faster (lower P50/P99 latency).

